# Prior-Position Study: Base Rates

For scholars now holding ladder-rank positions at US universities in computational
social science and adjacent fields, what position did they hold immediately before?

Two entry paths are compared:

- **Path A** — ladder-rank post at a non-US institution → US ladder-rank
- **Path B** — non-ladder US post (adjunct, lecturer, research scientist, project
  scientist, academic coordinator, extended postdoc) → US ladder-rank

The output is **base rates, not a causal estimate**. See the limitations section at
the end before reading anything into the numbers.

**Inputs:** `data/coded_trajectories.csv` (stage 3) and, for the agreement rate, a
hand-coded validation sample (stage 4).

> If you have not yet run stages 1–2 (they need network access to OpenAlex and
> ORCID), set `CODED = 'data/coded_fixture.csv'` below to render the notebook
> against synthetic fixture data. Those numbers are **meaningless** — they exist
> only to prove the tables render.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

# Notebook lives in stage5_analysis/; the package root is its parent.
ROOT = Path.cwd()
if not (ROOT / 'common.py').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from stage5_analysis.base_rates import (
    load, table_composition, table_base_rates, table_by_pubs,
    table_feeder_countries, table_gap_years, wilson,
)

CODED = ROOT / 'data' / 'coded_trajectories.csv'
if not CODED.exists():
    CODED = ROOT / 'data' / 'coded_fixture.csv'
    print('!! Real coded data not found; falling back to SYNTHETIC FIXTURES.')
    print('!! Every number below is meaningless until stages 1-2 have been run.')

pd.set_option('display.width', 120)
print(f'Reading {CODED}')

In [ ]:
df = load(CODED, exclude_flagged=False)

print(f"n with a coded prior position : {len(df)}")
print(f"flagged for manual review     : {(df['needs_manual_review'] == '1').sum()}")
print(f"appointment years             : {int(df['appointment_year'].min())}-{int(df['appointment_year'].max())}")
print()
print('By field:')
print(df['field'].value_counts().to_string())
print()
print('Appointment year source (ORCID start date vs OpenAlex proxy):')
print(df['appointment_year_source'].value_counts().to_string())

## 1. What did people hold immediately before?

In [ ]:
table_composition(df)

## 2. Base rates: Path A vs Path B

Three quantities, because "non-ladder" admits more than one reading:

- **Path B strict** — US non-ladder *academic* post (adjunct, lecturer, research
  scientist, project scientist, academic coordinator).
- **Path B broad** — the above, plus postdocs held for at least
  `extended_postdoc_min_years` (default 4), matching the brief's "extended postdoc".
- **Reference: standard US postdoc** — the modal US route. Kept separate because
  pooling it into Path B would swamp the comparison with the ordinary case.

`share` is the proportion of all coded appointments; `ci_low`/`ci_high` are Wilson
95% intervals. Field-level cells are small — read the intervals, not the points.

In [ ]:
base = table_base_rates(df)
base[base['group'].str.contains('ALL')]

In [ ]:
# By field, to confirm no single discipline is driving the pooled rate.
base[~base['group'].str.contains('ALL')]

## 3. Conditioned on pre-move publications

The crux question: does a strong publication record close the gap for the
non-ladder path, or does the structural signal dominate regardless of output?

Read **across** publication bands within a path. If Path B's share rises with
output while Path A's stays flat, that is consistent with publications
substituting for structural position — *consistent with*, not evidence of.
This is a cross-tabulation: people with 20+ pre-move works differ from those
with fewer in career stage, field norms and much else.

In [ ]:
table_by_pubs(df)

## 4. Which non-US systems feed US ladder-rank hiring

`share` is of all coded appointments; `share_of_path_a` is of Path A moves only.
Hong Kong is called out separately per the brief — check the printed `n` before
quoting any Hong Kong rate, since the cell is likely to be small.

In [ ]:
feeders = table_feeder_countries(df)
feeders

In [ ]:
hk = feeders[feeders['group'] == 'HK']
if hk.empty:
    print('No Hong Kong Path A moves observed in this sample.')
else:
    row = hk.iloc[0]
    lo, hi = wilson(int(row['count']), int(row['n']))
    print(f"Hong Kong -> US ladder-rank: {int(row['count'])} of {int(row['n'])} appointments")
    print(f"  {row['share']:.1%} of all appointments (95% CI {lo:.1%}-{hi:.1%})")
    print(f"  {row['share_of_path_a']:.1%} of all Path A moves")
    if int(row['count']) < 10:
        print('  CAUTION: n < 10. Report the count, not the rate.')

## 5. Years spent in the prior position

`gap_years` follows the brief's definition — appointment year minus prior position
*start* year. It measures time served in the prior post, not an employment gap.

In [ ]:
table_gap_years(df)

## 6. Validation: agreement rate

A random 50 from the automated output, hand-coded from CVs and faculty pages.
The brief's bar is ~85% agreement on `prior_rank_class`; below that the country
title mapping needs work before any base rate above is trustworthy.

To produce the inputs:

```
python -m stage4_validation.validate_coding export-sample --n 50
# hand-code data/validation_sample.csv, then:
python -m stage4_validation.validate_coding compute-agreement \
    --human-file data/validation_sample_coded.csv
```

In [ ]:
from stage4_validation.validate_coding import cohens_kappa

candidates = [
    ROOT / 'data' / 'validation_sample_coded.csv',
    ROOT / 'data' / 'validation_sample_fixture_coded.csv',
]
human_path = next((p for p in candidates if p.exists()), None)

if human_path is None:
    print('No hand-coded validation sample found. Run stage 4 first.')
else:
    if 'fixture' in human_path.name:
        print('!! Using SIMULATED hand-coding from the selftest.')
        print('!! This measures nothing about the title mapping.\n')
    machine = pd.read_csv(CODED, dtype=str).set_index('person_id')
    human = pd.read_csv(human_path, dtype=str)
    human = human[human['human_prior_rank_class'].notna()]

    pairs = [
        (machine.loc[pid, 'prior_rank_class'], hv.strip().lower())
        for pid, hv in zip(human['person_id'], human['human_prior_rank_class'])
        if pid in machine.index
    ]
    m = [p[0] for p in pairs]
    h = [p[1] for p in pairs]
    agree = sum(1 for x, y in zip(m, h) if x == y)
    rate = agree / len(pairs)

    print(f'n compared        : {len(pairs)}')
    print(f'percent agreement : {rate:.1%} ({agree}/{len(pairs)})')
    print(f"Cohen's kappa     : {cohens_kappa(m, h):.3f}")
    print()
    print('PASS: base rates can be reported.' if rate >= 0.85
          else 'FAIL: fix the title mapping before reporting base rates.')

In [ ]:
# Where the automated coding is least certain, by national system.
# Useful for targeting title-mapping work regardless of the agreement rate.
pd.crosstab(df['coding_system'], df['coding_confidence'])

## Limitations

**Selection.** People who take non-US posts and return differ systematically from
those who do not. These base rates describe the observed path; they do not
identify an effect of taking it.

**Survivorship.** The frame contains only people who landed a US ladder-rank job.
Everyone who tried either path and failed is invisible. This inflates the apparent
success of both paths, so the *comparison* survives but no absolute rate should be
read as a probability of success.

**Coverage.** ORCID skews younger and more quantitative, a real bias in a sample
spanning STS and digital humanities. Check `appointment_year_source` above: rows
resolved by the OpenAlex proxy rather than an ORCID start date carry a noisier
appointment year, biased late, because affiliation-by-year is derived from
publication metadata and papers in the pipeline still carry the old institution.

**Title ambiguity.** Rank class is inferred from title plus employing country. The
known soft spots are UK fixed-term lectureships (coded ladder, sometimes wrongly),
German *wissenschaftlicher Mitarbeiter* (doctoral vs postdoc), and "Research
Fellow" everywhere. The confidence crosstab above locates them; the agreement rate
bounds them.

**Transitional posts.** A one-year visiting position immediately before a hire is
coded as the prior position even when a longer substantive post preceded it. Those
rows are flagged `needs_manual_review` with the masked position named in
`coding_note`; re-run with `--exclude-flagged` to see how much they move the rates.

**Descriptive only.** No causal claim is made or supported anywhere in this notebook.